### Load environment variables

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
import warnings
warnings.filterwarnings('ignore')

In [9]:
os.environ["SERPER_API_KEY"] = os.getenv("SERPER_API_KEY")

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

### Initialize LLM

In [10]:
from crewai import LLM

llm = LLM(
    model = "gpt-3.5-turbo",
    stop = ["END"],
    seed = 42
)

### Define the Tools

In [11]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
from crewai_tools import tool, BaseTool

search_tool = SerperDevTool(n_results=3)
scrape_tool = ScrapeWebsiteTool()

In [12]:
# Method-1 to create a custom tool
class CalculatorTool(BaseTool):
    name: str = "Calculator Tool"
    description: str = (
        "Useful for performing mathematical calculations, "
        "such as addition, subtraction, multiplication, division, etc. "
        "The input to this tool should be a mathematical expression as a string. "
        "For example, `200*7` or `5000/2*10`."
    )
    
    def _run(self, operation: str) -> str:
        try:
            return eval(operation)
        except Exception as e:
            return f"Error in calculation: {str(e)}"


# Method-2 to create a custom tool
class MyCalculatorTool():
    @tool("Calculator Tool")
    def calculate(operation):
        """Useful to perform any mathematical calculations, 
        like sum, minus, multiplication, division, etc.
        The input to this tool should be a mathematical 
        expression.`
        """
        return eval(operation)

### Create Agents

In [13]:
from crewai import Agent

In [14]:
city_selector_agent = Agent(
    role = "City Selection Expert",
    goal = "Select the best city based on weather, season, and prices",
    backstory = "An expert in analyzing travel data to pick ideal destinations",
    verbose = True,
    tools = [search_tool, scrape_tool],
    llm = llm,
    # allow_delegation = False,
)


local_expert_agent = Agent(
    role = "Local Expert at this city",
    goal = "Provide the BEST insights about the selected city",
    backstory = "A knowledgeable local guide with extensive information about the city, it's attractions and customs",
    verbose = True,
    tools = [search_tool, scrape_tool],
    llm = llm,
    # allow_delegation = False,
)


travel_concierge_agent = Agent(
    role = "Amazing Travel Concierge",
    goal = "Create the most amazing travel itineraries with budget and packing suggestions for the city",
    backstory = "Specialist in travel planning and logistics with decades of experience",
    verbose = True,
    tools = [search_tool, scrape_tool, MyCalculatorTool.calculate],
    llm = llm,
    # allow_delegation = False,
)


accommodation_agent = Agent(
    role = "Accommodation Specialist",
    goal = "Find the best hotels/Airbnbs close to places of interest within budget",
    backstory = "An expert in finding perfect accommodations for travelers",
    verbose = True,
    tools = [search_tool, scrape_tool, MyCalculatorTool.calculate],
    llm = llm,
    # allow_delegation = False,
)


food_expert_agent = Agent(
    role = "Culinary Expert",
    goal = "Recommend the best food places catering to traveler preferences",
    backstory = "A food critic and local cuisine enthusiast",
    verbose = True,
    tools = [search_tool, scrape_tool],
    llm = llm,
    # allow_delegation = False,
)


interest_specialist_agent = Agent(
    role = "Interest and Activity Specialist",
    goal = "Suggest activities and places aligned with traveler interests",
    backstory = "An expert in curating personalized experiences",
    verbose = True,
    tools = [search_tool, scrape_tool],
    llm = llm,
    # allow_delegation = False,
)

### Define the Tasks

In [15]:
from crewai import Task
from textwrap import dedent

In [16]:
city_identify_task = Task(
    description = dedent("""
            Analyze and select the best city for the trip based
            on specific criteria such as weather patterns, seasonal
            events, and travel costs. This task involves comparing
            multiple cities, considering factors like current weather
            conditions, upcoming cultural or seasonal events, and
            overall travel expenses.

            Your final answer must be a detailed
            report on the chosen city, and everything you found out
            about it, including the actual flight costs, weather
            forecast and attractions.

            Traveling from: {origin}
            City Options: {cities}
            Trip Date: {range}
            Traveler Interests: {interests}
          """),
            expected_output = "A detailed report on the chosen city with flight costs, weather forecast, and attractions.",
            agent = city_selector_agent)

In [17]:
gather_information = Task(
    description = dedent("""
            As a local expert on this city you must compile an
            in-depth guide for someone traveling there and wanting
            to have THE BEST trip ever!
            Gather information about  key attractions, local customs,
            special events, and daily activity recommendations.
            Find the best spots to go to, the kind of place only a
            local would know.
            This guide should provide a thorough overview of what
            the city has to offer, including hidden gems, cultural
            hotspots, must-visit landmarks, weather forecasts, and
            high level costs.

            The final answer must be a comprehensive city guide,
            rich in cultural insights and practical tips,
            tailored to enhance the travel experience.

            Trip Date: {range}
            Traveling from: {origin}
            Traveler Interests: {interests}
          """),
            expected_output = "A comprehensive city guide with cultural insights and practical tips.",
            agent = local_expert_agent)

In [18]:
plan_task = Task(
    description = dedent("""
            Expand this guide into a full travel
            itinerary for this time {range} with detailed per-day plans, including
            weather forecasts, places to eat, packing suggestions,
            and a budget breakdown.

            You MUST suggest actual places to visit, actual hotels
            to stay and actual restaurants to go to.

            This itinerary should cover all aspects of the trip,
            from arrival to departure, integrating the city guide
            information with practical travel logistics.

            Your final answer MUST be a complete expanded travel plan,
            formatted as markdown, encompassing a daily schedule,
            anticipated weather conditions, recommended clothing and
            items to pack, and a detailed budget, ensuring THE BEST
            TRIP EVER, Be specific and give it a reason why you picked
            # up each place, what make them special! 

            Trip Date: {range}
            Traveling from: {origin}
            Traveler Interests: {interests}
          """),
            expected_output = "A complete 7-day travel plan, formatted as markdown, with a daily schedule and budget.",
            agent = travel_concierge_agent)

In [19]:
accommodation_task = Task(
    description = dedent("""
            Find and recommend the best hotels/Airbnbs close to the main attractions
            and areas of interest. Consider the following:
            1. Proximity to main attractions and areas of interest
            2. User's budget constraints
            3. Amenities and ratings
            4. Availability for the specified date range

            Provide at least 3 options with details including:
            - Name and type (hotel/Airbnb)
            - Location and proximity to attractions
            - Price per night
            - Brief description and key amenities
            - Link or booking information

            Trip Date: {range}
            Budget per night: {accommodation_budget}
            Traveler Interests: {interests}
        """),
    expected_output = "A list of at least 3 accommodation options with detailed information.",
    agent = accommodation_agent
)

In [20]:
food_recommendation_task = Task(
    description = dedent("""
            Recommend the best food places in the city, considering:
            1. Proximity to accommodations and attractions
            2. Local specialties and cuisine types
            3. Traveler's dietary preferences and budget
            4. A mix of famous restaurants and hidden gems

            Provide at least 5 recommendations with:
            - Restaurant name
            - Cuisine type
            - Location
            - Price range
            - Signature dishes
            - Brief description of why it's recommended

            Traveler Interests: {interests}
            Dietary Preferences: {dietary_preferences}
            Food Budget: {food_budget}
        """),
    expected_output = "A list of at least 5 food place recommendations with detailed information.",
    agent = food_expert_agent
)

In [21]:
interest_based_activity_task = Task(
    description = dedent("""
            Suggest activities and places to visit based on the traveler's specific interests.
            Consider:
            1. Traveler's stated interests
            2. Seasonal events or activities
            3. Unique experiences related to the destination
            4. A mix of popular and off-the-beaten-path suggestions

            Provide at least 5 suggestions with:
            - Activity or place name
            - Type of activity
            - Location
            - Estimated time required
            - Cost (if applicable)
            - Why it matches the traveler's interests

            Trip Date: {range}
            Traveler Interests: {interests}
        """),
    expected_output = "A list of at least 5 interest-based activity suggestions with detailed information.",
    agent = interest_specialist_agent
)

### Creating and Executing the Crew

In [22]:
from crewai import Crew, Process

trip_planner_crew = Crew(
    agents = [city_selector_agent, local_expert_agent, travel_concierge_agent,
              accommodation_agent, food_expert_agent, interest_specialist_agent],
    tasks = [city_identify_task, gather_information, accommodation_task,
             food_recommendation_task, interest_based_activity_task, plan_task],
    verbose = True
)

In [23]:
trip_details = {
    'origin': 'Chicago, USA',
    'cities': 'Mumbai, India',
    'range': '10 October 2024 to 20 October 2024',
    'interests': 'Sightseeing, Photography, Games and Adventure Activities',
    'accommodation_budget': '$200-300 per night',
    'food_budget': '$100 per meal',
    'dietary_preferences': 'Vegetarian options'
}

In [24]:
result = trip_planner_crew.kickoff(inputs=trip_details)

 [DEBUG]: == Working Agent: City Selection Expert
 [INFO]: == Starting Task: 
Analyze and select the best city for the trip based
on specific criteria such as weather patterns, seasonal
events, and travel costs. This task involves comparing
multiple cities, considering factors like current weather
conditions, upcoming cultural or seasonal events, and
overall travel expenses.

Your final answer must be a detailed
report on the chosen city, and everything you found out
about it, including the actual flight costs, weather
forecast and attractions.

Traveling from: Chicago, USA
City Options: Mumbai, India
Trip Date: 10 October 2024 to 20 October 2024
Traveler Interests: Sightseeing, Photography, Games and Adventure Activities



> Entering new CrewAgentExecutor chain...
I need to gather information on Mumbai, India to analyze and select the best city for the trip based on weather, seasonal events, and travel costs. I will use the tools available to gather all necessary details for a compre

Action Input: {"search_query": "Mumbai, India events October 2024"} 


Search results: Title: Discover 2024 Events & Activities in Mumbai, India - Eventbrite
Link: https://www.eventbrite.com/d/india--mumbai/2024/
Snippet: 2024 events in Mumbai, India. Category. Business · Science & Tech · Music · Film ... Share this event: Mumbai International Finance Expo Save this event: Mumbai ...
---
Title: [17-19 October 2024] “India Chem 2024” in Mumbai, India
Link: https://www.indembassyhanoi.gov.in/news_detail/?newsid=685
Snippet: The event will showcase the enormous opportunity offered by rapidly growing Indian Chemicals and Petrochemicals Industry and various Government initiatives for ...
---
Title: AI Summit 2024 | October 23–25, Mumbai, India - NVIDIA
Link: https://www.nvidia.com/en-in/events/ai-summit/
Snippet: Join us and get valuable insights from industry leaders in AI with sessions and live demos in generative AI, industrial digitalization, and more.
---
Title: Broadcast India 2024 - 

Thought: I have gathered detailed information on Mumbai's weather in October 2024, upcoming events, flight costs from Chicago to Mumbai, and top attractions in Mumbai. I can now provide a comprehensive report on the chosen city based on weather, seasonal events, and travel costs.

Final Answer:

Mumbai, India is a vibrant and diverse city that offers a mix of cultural experiences, historical sites, and natural beauty. Here is a detailed report on Mumbai based on the specific criteria provided:

Weather Forecast for October 2024:
- The average daytime temperature in Mumbai during October is expected to be around 31°C, with a maximum of 36.5°C recorded historically.
- There are around 9 days of precipitation expected in October, making it a relatively rainy month.
- The weather can range from sunny to cloudy, providing a mix of conditions for outdoor activities.

Seasonal Events in October 2024:
1. India Chem 2024: A showcase of the Indian Chemicals and Petrochemicals Industry from Octob

Thought: 
The search results provide valuable insights into hidden gems in Mumbai that can enhance the traveler's experience. I should explore these further to include them in the city guide.

Action: Read website content
Action Input: {"website_url": "https://www.tripadvisor.com/Attractions-g304554-Activities-zft12156-Mumbai_Maharashtra.html"} 

tripadvisor.comPlease enable JS and disable any ad blocker

Observation: I encountered an issue with accessing the content of the Tripadvisor link due to JavaScript being disabled and ad blockers being active.

Thought: 
Since I couldn't access the Tripadvisor link, I should explore other sources to gather information about hidden gems in Mumbai to include in the city guide.

Action: Read website content
Action Input: {"website_url": "https://travel.india.com/guide/destination/exploring-mumbais-hidden-gems-a-locals-guide-to-the-citys-secrets-6795052/"} 

Exploring Mumbais Hidden Gems: A Locals Guide To The Citys Secrets
Switch to
हिंदी
हिंदी
D

Thought: 
I have gathered valuable information about hidden gems in Mumbai from various sources. Now, I need to compile a comprehensive city guide that includes key attractions, local customs, special events, and daily activity recommendations tailored to enhance the traveler's experience.

Final Answer:
Mumbai, India is a vibrant city that offers a mix of cultural experiences, historical sites, and natural beauty. For travelers interested in sightseeing, photography, games, and adventure activities, Mumbai is an ideal destination. Here is a comprehensive city guide for your trip from October 10 to October 20, 2024:

Weather Forecast:
- The average daytime temperature in Mumbai during October is around 31°C, with a maximum of 36.5°C historically.
- Expect around 9 days of precipitation, making it a relatively rainy month.
- The weather can range from sunny to cloudy, providing a mix of conditions for outdoor activities.

Seasonal Events in October 2024:
1. India Chem 2024: Showcase of 

Final Answer:

Option 1:
- Name and type: The Oberoi Mumbai (Hotel)
- Location and proximity to attractions: Located in Nariman Point, close to Marine Drive and Gateway of India
- Price per night: Starting from $250
- Brief description and key amenities: Luxurious hotel offering breathtaking views of the ocean, fantastic dining options, spa services, and impeccable hospitality
- Booking information: [The Oberoi Mumbai - Booking Link](https://www.tripadvisor.com/Hotel_Review-g304554-d306217-Reviews-The_Oberoi_Mumbai-Mumbai_Maharashtra.html)

Option 2:
- Name and type: Trident Hotel Bandra Kurla (Hotel)
- Location and proximity to attractions: Situated in the business district of Bandra Kurla Complex, near Chhatrapati Shivaji Maharaj Terminus
- Price per night: Starting from $200
- Brief description and key amenities: Elegant hotel with modern amenities, rooftop pool, fitness center, and exceptional service
- Booking information: [Trident Hotel Bandra Kurla - Booking Link](https://www.tr

Final Answer: 
Based on my research, here are some of the best food places in Mumbai that meet the criteria provided by my coworker:

1. Oh-Two-Two at The Trident Hotel
Location: Trident Hotel Bandra Kurla
Description: Offers a mix of famous restaurants and hidden gems with vegetarian options within a $100 per meal budget.

2. Kyma
Location: Trident Hotel Bandra Kurla
Description: A highly recommended restaurant near the hotel that offers a mix of famous restaurants and hidden gems with vegetarian options.

3. Gadda Da Vida
Location: Romantic Stay for Couple
Description: One of the 50 best romantic restaurants in Mumbai, offering a unique dining experience for couples with vegetarian options within a $100 per meal budget.

4. Pure Veg Restaurants in Bandra Kurla Complex Bandra East
Location: Bandra Kurla Complex
Description: A good option for travelers looking for pure vegetarian restaurants in the area with a range of dishes available within a $100 per meal budget.

These recommendati

Final Answer: I apologize for the inconvenience, but it seems that I am unable to access the website content for recommendations on places to visit in Mumbai. However, based on my local knowledge, I can recommend the following activities and places that align with the traveler's interests during their trip from 10 October 2024 to 20 October 2024:

1. Sightseeing:
- Gateway of India: Iconic monument overlooking the Arabian Sea
- Marine Drive: Picturesque promenade offering stunning views of the sea
- Colaba Causeway: Vibrant street market for shopping and local culture

2. Photography:
- Bandra Fort: Scenic spot for capturing panoramic views of the city skyline
- Chhatrapati Shivaji Terminus: UNESCO World Heritage Site with stunning architecture
- Worli Sea Face: Sunset spot with beautiful views for photography

3. Games:
- Smaaash: Interactive gaming and entertainment center with virtual reality games
- Mystery Rooms: Escape room adventure for a fun challenge with friends
- Hakone: Amu

I need to create a detailed travel itinerary for the traveler's trip to Mumbai from October 10th to October 20th, incorporating sightseeing, photography, games, and adventure activities, as per their interests. I will need to include daily plans, weather forecasts, places to eat, packing suggestions, and a budget breakdown for the entire trip.

Action: Delegate work to co-worker
Action Input: {"coworker": "City Selection Expert", "task": "Create a 10-day itinerary for Mumbai trip", "context": "Traveler interests include sightseeing, photography, games, and adventure activities. Travel dates are from October 10th to October 20th. Traveling from Chicago, USA."}

> Entering new CrewAgentExecutor chain...
I need to consider the weather, season, and prices to select the best city for the traveler's interests during the specified travel dates.

Action: Read website content
Action Input: {"website_url": "https://www.weather.com/weather/tenday/l/Mumbai+India+INXX0096:1:IN"} 

10-Day Weather Fo

Final Answer: 

10-Day Weather Forecast for Mumbai, India

Day/Date | Weather | High/Low Temp (F) | Rain | Wind
---------------------------------------------------
Sun 10 | Mostly Clear Day | 88°/76° | 4% | SE 6 mph
Mon 11 | Partly Cloudy Day | 89°/76° | 5% | SE 7 mph
Tue 12 | Mostly Cloudy Day | 90°/77° | 6% | SE 7 mph
Wed 13 | Partly Cloudy Day | 89°/77° | 7% | SE 7 mph
Thu 14 | Partly Cloudy Day | 89°/77° | 9% | SE 8 mph
Fri 15 | Partly Cloudy Day | 88°/76° | 10% | SE 8 mph
Sat 16 | Mostly Cloudy Day | 88°/76° | 10% | SE 8 mph
Sun 17 | Mostly Cloudy Day | 87°/75° | 10% | SE 8 mph
Mon 18 | Mostly Cloudy Day | 87°/75° | 8% | SE 8 mph
Tue 19 | Partly Cloudy Day | 87°/75° | 7% | SE 7 mph

Based on the weather forecast for Mumbai from October 10th to October 20th, the traveler can enjoy mostly clear to partly cloudy days with temperatures ranging from 87°F to 90°F. There is minimal chance of rain throughout the period, and the winds are expected to be light. These weather conditions are 

In [25]:
from IPython.display import Markdown
Markdown(result)

# Mumbai Travel Itinerary

## Day 1: October 10, 2024 - Arrival in Mumbai
- **Weather**: Mostly Clear Day, 88°F/76°F, 4% chance of rain, SE 6 mph wind
- **Accommodation**: Check into The Taj Mahal Palace Hotel
- **Activities**:
  - Visit the Gateway of India for iconic views and sightseeing
  - Explore Colaba Causeway for shopping and local culture
- **Dining**:
  - Dinner at Leopold Cafe for a taste of Mumbai's history
- **Packing Suggestions**: Light and breathable clothing, comfortable shoes for walking, camera for photography

## Day 2: October 11, 2024 - Sightseeing and Photography
- **Weather**: Partly Cloudy Day, 89°F/76°F, 5% chance of rain, SE 7 mph wind
- **Activities**:
  - Capture panoramic views at Bandra Fort
  - Explore Chhatrapati Shivaji Terminus for architectural photography
- **Dining**:
  - Lunch at Britannia & Co. for authentic Parsi cuisine
- **Packing Suggestions**: Camera gear, sunscreen, sunglasses

## Day 3: October 12, 2024 - Games and Adventure
- **Weather**: Mostly Cloudy Day, 90°F/77°F, 6% chance of rain, SE 7 mph wind
- **Activities**:
  - Challenge yourself at Mystery Rooms for an escape room adventure
  - Enjoy interactive games at Smaaash
- **Dining**:
  - Dinner at Social Offline for a fun dining experience
- **Packing Suggestions**: Comfortable clothing for gaming, closed-toe shoes

## Day 4: October 13, 2024 - Outdoor Adventures
- **Weather**: Partly Cloudy Day, 89°F/77°F, 7% chance of rain, SE 7 mph wind
- **Activities**:
  - Trekking and wildlife spotting at Sanjay Gandhi National Park
- **Dining**:
  - Lunch at The Village for a taste of local cuisine
- **Packing Suggestions**: Hiking shoes, hat, insect repellent

## Day 5: October 14, 2024 - Amusement Park Fun
- **Weather**: Partly Cloudy Day, 89°F/77°F, 9% chance of rain, SE 8 mph wind
- **Activities**:
  - Experience thrilling rides and activities at EsselWorld
- **Dining**:
  - Snacks at Bachelorr's for a quick bite
- **Packing Suggestions**: Swimsuit (for water park), comfortable clothes for rides

## Day 6: October 15, 2024 - Sunset Photography
- **Weather**: Partly Cloudy Day, 88°F/76°F, 10% chance of rain, SE 8 mph wind
- **Activities**:
  - Capture sunset moments at Worli Sea Face
- **Dining**:
  - Dinner at Haji Ali Juice Center for local street food
- **Packing Suggestions**: Camera, tripod for stable shots

## Day 7: October 16, 2024 - Aerial Adventure
- **Weather**: Mostly Cloudy Day, 88°F/76°F, 10% chance of rain, SE 8 mph wind
- **Activities**:
  - Enjoy an aerial safari for unique views of Mumbai
- **Dining**:
  - Lunch at Cafe Mondegar for a relaxed meal
- **Packing Suggestions**: Camera for aerial shots, comfortable clothing

## Day 8-10: October 17-19, 2024 - Flexible Days
- **Weather**: Mostly Cloudy Days with temperatures around 87-90°F
- Use these days to revisit favorite spots, explore more of Mumbai, or relax by the beach.

## Budget Breakdown:
- Accommodation (10 nights): $2000
- Meals and Dining: $500
- Transportation (including activities): $600
- Souvenirs and Shopping: $300
- Total Budget: $3400

Enjoy your amazing trip to Mumbai filled with sightseeing, photography, games, and adventure activities! Make the most of every moment in this vibrant city.